# Análise de Reembolso de Despesas

Este notebook demonstra como criar agentes que utilizam plugins para processar despesas de viagem a partir de imagens locais de recibos, gerar um email de pedido de reembolso de despesas e visualizar os dados das despesas usando um gráfico de pizza. Os agentes escolhem dinamicamente as funções com base no contexto da tarefa.

Passos:
1. O Agente OCR processa a imagem local do recibo e extrai os dados das despesas de viagem.
2. O Agente de Email gera um email de pedido de reembolso de despesas.

### Exemplo de um cenário de despesas de viagem:
Imagine que é um funcionário a viajar para uma reunião de negócios noutra cidade. A sua empresa tem uma política de reembolso de todas as despesas razoáveis relacionadas com a viagem. Eis uma descrição das possíveis despesas de viagem:
- Transporte:
Passagem aérea de ida e volta da sua cidade de origem até à cidade de destino.
Táxi ou serviços de transporte por aplicação para ir e vir do aeroporto.
Transporte local na cidade de destino (como transporte público, aluguer de carros ou táxis).

- Alojamento:
Estadia de hotel por três noites num hotel de negócios médio perto do local da reunião.

- Refeições:
Subsídio diário para refeições de pequeno-almoço, almoço e jantar, baseado na política de diárias da empresa.

- Despesas Diversas:
Taxas de estacionamento no aeroporto.
Custos de acesso à Internet no hotel.
Gorjetas ou pequenas despesas de serviço.

- Documentação:
Submete todos os recibos (voos, táxis, hotel, refeições, etc.) e um relatório de despesas preenchido para reembolso.


## Importar bibliotecas necessárias

Importe as bibliotecas e módulos necessários para o notebook.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import Content, Message, tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Definir Modelos de Despesas

 Crie um modelo Pydantic para despesas individuais e uma classe ExpenseFormatter para converter uma consulta do utilizador em dados de despesa estruturados.

 Cada despesa será representada no formato:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Definindo Ferramentas - Gerar o Email

Crie uma função de ferramenta para gerar um email para submeter uma declaração de despesas.
- Esta ferramenta usa o decorador `@tool` do Microsoft Agent Framework.
- Calcula o montante total das despesas e formata os detalhes num corpo de email.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Carregar Imagens de Recibos de Despesas de Viagem

Crie uma função auxiliar para carregar imagens de recibos como conteúdo multimodal nativo.
- Lê os bytes da imagem do recibo e envolve-os no Agent Framework `Content` para o agente analisar.


In [ ]:
def load_receipt_image(image_path: str = "receipt.jpg") -> Content:
    """Load a receipt image as native multimodal content."""
    with open(image_path, "rb") as f:
        image_bytes = f.read()
    return Content.from_data(image_bytes, "image/jpeg")

## Processar Despesas

Defina os agentes e integre-os num fluxo de trabalho sequencial utilizando `WorkflowBuilder`.
- O agente OCR extrai dados estruturados de despesas a partir da imagem do recibo fornecida na mensagem do utilizador.
- O agente de Email pega nos dados extraídos e gera um email profissional de pedido de reembolso de despesas utilizando a ferramenta `generate_expense_email`.
- `WorkflowBuilder` com `add_edge` cria uma pipeline sequencial: Agente OCR → Agente de Email.


In [ ]:
ocr_agent = client.as_agent(
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Analyze the receipt image supplied in the user message and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Função principal

Construa o fluxo de trabalho sequencial e execute-o para processar a imagem do recibo e gerar o e-mail de pedido de reembolso.


> **Nota:** Os bytes do recibo estão encapsulados no multimodal nativo do Agent Framework `Content`, pelo que o Foundry os recebe como entrada de imagem em vez de texto normal.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
receipt_message = Message(
    role="user",
    contents=[prompt, load_receipt_image("receipt.jpg")],
)

events = workflow.run(
    receipt_message,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
